# 🏒 Lab 01 · Teach a computer to predict, then let it plan by imagining

**World Models course · Lectures 3 and 12 · HW3 core** &nbsp;|&nbsp; ⏱ about 60 min &nbsp;|&nbsp; 💻 CPU only (no GPU needed)

Imagine a puck sliding on an ice rink. You can give it small pushes. By the end of this lab you will have:

1. **Collected experience:** watched the puck move under random pushes.
2. **Learned a world model:** a small formula that guesses *"if I push like this, where will the puck be next?"*
3. **Planned with imagination:** tried hundreds of push sequences *inside the model* and picked the best one, like a chess player thinking ahead.

This loop of **predict → imagine → choose → act** is what systems such as DeepMind's MuZero and Dreamer and Meta's V-JEPA 2-AC robot planner run at a much larger scale.

### How this lab works
| Symbol | Meaning |
|---|---|
| 🧩 **Challenge** | A tiny blank `___` for you to fill in. Usually one short expression capturing the key idea. |
| 🔮 **Predict** | Click an answer *before* running the next cell. Guessing wrong is how you learn. |
| 🎛️ **Playground** | Sliders. Drag them and watch what changes. |
| 🏭 **Industry link** | Where this idea shows up in real products and research labs. |

If you leave a blank empty or get it wrong, the cell explains why and **uses a working answer so the rest of the notebook still runs.** Nothing breaks. Each challenge has a hint and an answer dropdown underneath.

> **In Colab:** use *Runtime → Run all* for a quick tour, or run cells one at a time with **Shift + Enter** (recommended).

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
    _WIDGETS = True
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
DT = 0.12

def _ref_true_step(state, push):
    position, velocity = state[..., :2], state[..., 2:]
    new_velocity = 0.90 * velocity + 0.16 * push - 0.015 * position**3
    new_position = position + DT * new_velocity
    return np.concatenate([new_position, new_velocity], axis=-1)

_r = np.random.default_rng(0)
CHALLENGES["true_step"] = dict(
    title="Move the puck", reference=_ref_true_step,
    cases=[(_r.normal(size=4), _r.uniform(-1, 1, 2)), (_r.normal(size=(5, 4)), _r.uniform(-1, 1, (5, 2)))],
    hint="Distance = speed × time. The new position is the old position plus (new velocity × DT).",
    why="This one line is <b>Euler integration</b>: nudge position along velocity for a small time step. Game and robot physics engines repeat it thousands of times per second.")

def _ref_pairs(states, pushes, episode_ids):
    now = states[episode_ids, :-1].reshape(-1, 4)
    nxt = states[episode_ids, 1:].reshape(-1, 4)
    act = pushes[episode_ids].reshape(-1, 2)
    return now, act, nxt - now

def _test_pairs(fn):
    s = _r.normal(size=(6, 5, 4)); a = _r.normal(size=(6, 4, 2)); ids = np.array([0, 3, 4])
    got, ref = fn(s, a, ids), _ref_pairs(s, a, ids)
    ok = all(np.allclose(g, r) for g, r in zip(got, ref))
    return ok, "The third output should be the <i>change</i> in state from one step to the next."
CHALLENGES["pairs"] = dict(title="What should the model predict?", reference=_ref_pairs, test=_test_pairs,
    hint="We want the model to learn how much the state <i>changes</i>: next state minus current state.",
    why="Predicting the <b>change</b> (a “delta”) is easier than predicting the whole next state, because most of the next state is just the current state. Almost every learned dynamics model does this.")

def _ref_imagine(model_step, start_state, pushes):
    state = np.array(start_state, dtype=float)
    path = [state]
    for push in pushes:
        state = model_step(state, push)
        path.append(state)
    return np.array(path)
def _test_imagine(fn):
    pushes = _r.uniform(-1, 1, (7, 2)); start = np.array([1., -1., 0., 0.])
    ok = np.allclose(fn(_ref_true_step, start, pushes), _ref_imagine(_ref_true_step, start, pushes))
    return ok, "Each step should feed the model's <i>own previous guess</i> back into the model."
CHALLENGES["imagine"] = dict(title="Imagine the future", reference=_ref_imagine, test=_test_imagine,
    hint="Inside the loop, replace the state with what the model predicts from the current state and push.",
    why="This is a <b>rollout</b>. The model never sees the real world here. It builds on its own guesses, like daydreaming.")

CHALLENGES["pick_best"] = dict(title="Pick the best imagined plan", reference=lambda costs: int(np.argmin(costs)),
    cases=[np.array([3., 1., 2.]), np.array([0.5, 9., -1., 4.])],
    hint="Lower cost is better. Which numpy function returns the <i>position</i> of the smallest value?",
    why="<code>np.argmin</code> returns the index of the lowest cost. This “try many, keep the best” approach is called <b>random shooting</b>.")

CHALLENGES["first_push"] = dict(title="Act, then re-plan", reference=lambda best_plan: best_plan[0],
    cases=[np.arange(24.).reshape(12, 2)],
    hint="The plan is a list of pushes for the next several steps. We only carry out the very first one.",
    why="Doing only the first action and then planning again from the <i>real</i> new state is <b>Model Predictive Control (MPC)</b>. It corrects the model's mistakes every step.")

def _ref_elites(plans, costs, n_elite):
    return plans[np.argsort(costs)[:n_elite]]
CHALLENGES["elites"] = dict(title="Keep the elite plans", reference=_ref_elites,
    cases=[(np.arange(10.)[:, None, None] * np.ones((10, 3, 2)), np.array([5, 2, 8, 1, 9, 0, 7, 3, 6, 4.]), 3)],
    hint="<code>np.argsort(costs)</code> lists indices from lowest to highest cost. Keep the first <code>n_elite</code> of them.",
    why="The <b>Cross-Entropy Method (CEM)</b> keeps the best plans and samples new ones near them. Meta's V-JEPA 2-AC uses CEM to plan robot-arm motions.")

QUIZZES["split"] = dict(q="Why do we split whole episodes into train/test instead of shuffling individual time steps?",
    options=["Shuffling is slower", "Neighbouring steps are almost identical, so test steps would leak into training", "scikit-learn requires episodes"],
    answer=1, explain="Step 17 and step 18 of the same episode are nearly copies. If one is in training and the other in test, the test is too easy and you fool yourself. Always split by <b>whole episode</b>.")
QUIZZES["compound"] = dict(predict=True, q="The simple model is very accurate one step ahead. If it predicts 20 steps ahead by feeding in its own guesses, the error will…",
    options=["stay about the same", "grow, because small mistakes compound", "shrink, because errors average out"],
    answer=1, explain="Each guess starts from the previous guess, so small errors snowball. This <b>compounding error</b> is the central difficulty of world models.")
QUIZZES["mpc"] = dict(q="Why execute only the first push and then re-plan, instead of running the whole 12-step plan?",
    options=["It is faster to compute", "The model is imperfect, so we correct course using the real new state", "The robot can only remember one action"],
    answer=1, explain="Like a satnav re-routing after every turn: re-planning from the <b>real</b> state stops model errors from piling up.")
QUIZZES["perfect"] = dict(predict=True, q="With re-planning every step, how will CEM with the <i>simple</i> model compare with CEM using the <i>true</i> physics?",
    options=["Much worse: about double the cost", "About the same", "Clearly better"],
    answer=1, explain="Re-planning from the <b>real</b> state every step corrects most of the simple model's small mistakes, so closed-loop MPC is surprisingly forgiving.")
QUIZZES["blind"] = dict(predict=True, q="Now plan 35 pushes once and execute them blindly with no re-planning. Which model's plan ends farther from the centre?",
    options=["Simple model", "Rich model", "Both end in the same place"],
    answer=0, explain="Without feedback, every error in imagination becomes an error in the real world. The simple model does not know about the bowl force, so its long plan drifts.")
print('✅ Setup complete. Scroll down and run the cells in order.')

---
## 1 · Meet the world 🌍

The puck's **state** is 4 numbers: position `(x, y)` and velocity `(vx, vy)`. The **action** is a push `(px, py)`, each between -1 and 1.

Each time step (`DT = 0.12` seconds):
* **Velocity:** 90% of the old velocity survives (ice friction), the push adds some, and the rink is slightly bowl-shaped, so the puck is pulled back toward the centre. That pull is the `-0.015 · position³` term, and it is stronger far from the centre.
* **Position:** moves along the new velocity.

### 🧩 Challenge 1 · Move the puck
Fill in the one line that moves the position forward.

In [ ]:
DT = 0.12   # seconds per step

def true_step(state, push):
    # state[..., :2] = position (x, y); state[..., 2:] = velocity (vx, vy)
    # The "..." means "works for one state OR a whole batch of states at once".
    position, velocity = state[..., :2], state[..., 2:]

    new_velocity = 0.90 * velocity + 0.16 * push - 0.015 * position**3   # friction + push + bowl
    new_position = ___                            # 🧩 move along the velocity

    return np.concatenate([new_position, new_velocity], axis=-1)   # glue back into 4 numbers

true_step = check("true_step", true_step)

<details><summary>🤔 <b>Need a hint?</b></summary>

Distance travelled in one step = velocity × time. Add that to where you were.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>new_position = position + DT * new_velocity                            # 🧩 move along the velocity</pre>

</details>

### 🎛️ Playground · Drive the puck yourself
Pick a constant push and watch 40 steps unfold. Try a strong push to the right, then a diagonal one.
**Notice:** the puck never flies off forever. The bowl-shaped rink bends it back.

In [ ]:
def drive(push_x=0.6, push_y=0.0, steps=40):
    state = np.array([-1.5, -1.0, 0.0, 0.0])        # start bottom-left, at rest
    path = [state]
    for _ in range(steps):
        state = true_step(state, np.array([push_x, push_y]))
        path.append(state)
    path = np.array(path)

    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    ax.plot(path[:, 0], path[:, 1], "-o", ms=3, label="puck path")
    ax.plot(*path[0, :2], "gs", ms=10, label="start")
    ax.plot(0, 0, "r*", ms=15, label="rink centre")
    ax.set(xlim=(-3, 3), ylim=(-3, 3), xlabel="x", ylabel="y", title=f"Constant push ({push_x:+.1f}, {push_y:+.1f})")
    ax.legend(loc="upper left", fontsize=8); plt.show()

playground(drive, push_x=(-1.0, 1.0, 0.1, 0.6), push_y=(-1.0, 1.0, 0.1, 0.0), steps=(5, 120, 5, 40))

---
## 2 · Collect experience 📼

A world model learns from **recorded experience**. We let the puck wander under *random* pushes for 100 **episodes** (runs) of 45 steps each.

The result is two arrays:
| Array | Shape | Read it as |
|---|---|---|
| `states` | (100, 46, 4) | episode → time step → the 4 state numbers |
| `pushes` | (100, 45, 2) | episode → time step → the push given at that step |

There is one more state than push per episode because the final state has no push after it.

In [ ]:
rng = np.random.default_rng(17)          # fixed seed, so everyone gets the same "random" data

def collect(n_episodes=100, length=45):
    all_states, all_pushes = [], []
    for _ in range(n_episodes):
        state = np.r_[rng.uniform(-1.5, 1.5, 2), rng.uniform(-0.3, 0.3, 2)]   # random start
        states, pushes = [state], []
        for t in range(length):
            push = rng.uniform(-1, 1, 2)        # a random push: we are just exploring
            state = true_step(state, push)
            states.append(state); pushes.append(push)
        all_states.append(states); all_pushes.append(pushes)
    return np.array(all_states), np.array(all_pushes)

states, pushes = collect()
print("states:", states.shape, " pushes:", pushes.shape)

# Split WHOLE episodes: 70 to learn from, 15 to tune on, 15 kept secret for the final test.
order = rng.permutation(len(states))
train_ids, val_ids, test_ids = order[:70], order[70:85], order[85:]
print("train / validation / test episodes:", len(train_ids), len(val_ids), len(test_ids))

fig, ax = plt.subplots(figsize=(4.8, 4.8))
for ep in train_ids[:12]:
    ax.plot(states[ep, :, 0], states[ep, :, 1], alpha=0.7)
ax.set(title="12 random-push episodes", xlabel="x", ylabel="y"); plt.show()

In [ ]:
quiz("split")

---
## 3 · Learn a world model 🧠

A **world model** here is a function: *(current state, push) → predicted next state*.

We use **linear regression**, the simplest learner there is: it finds the best weights so that
`change ≈ weights × inputs`. No neural network is needed yet, which keeps everything transparent.

### 🧩 Challenge 2 · What should the model predict?
Build the training data. Inputs are the current state and the push. Fill in the **target** the model should learn to output.

In [ ]:
def make_training_pairs(states, pushes, episode_ids):
    now  = states[episode_ids, :-1].reshape(-1, 4)    # every state except the last in each episode
    nxt  = states[episode_ids, 1:].reshape(-1, 4)     # the state one step later
    push = pushes[episode_ids].reshape(-1, 2)          # the push that caused the transition
    target = ___                               # 🧩 what should the model learn to output?
    return now, push, target

make_training_pairs = check("pairs", make_training_pairs)

<details><summary>🤔 <b>Need a hint?</b></summary>

The model should output how much each of the 4 numbers <i>changes</i> in one step.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>target = nxt - now                               # 🧩 what should the model learn to output?</pre>

</details>

### Two models, different "hints"
* **Simple model:** sees only `state` and `push`. It can't represent the bowl's `position³` pull.
* **Rich model:** also gets `position³` as an extra input feature. Features are hints we hand to the learner.

We compare both with a **do-nothing baseline** that predicts *"the state won't change"*. A model that can't beat that isn't learning anything.

In [ ]:
def features(state, push, rich):
    base = np.concatenate([state, push], axis=-1)                        # 6 numbers
    return np.concatenate([base, state[..., :2]**3], axis=-1) if rich else base   # +2 cubic hints

def learn_model(rich):
    now, push, change = make_training_pairs(states, pushes, train_ids)
    reg = Ridge(alpha=1e-3).fit(features(now, push, rich), change)    # find the best weights
    W, b = reg.coef_, reg.intercept_                                   # the whole model is a matrix + a vector!
    def model_step(state, push):
        return state + features(state, push, rich) @ W.T + b          # state + predicted change
    return model_step, W

simple_step, W_simple = learn_model(rich=False)
rich_step, W_rich = learn_model(rich=True)

now, push, change = make_training_pairs(states, pushes, test_ids)
real_next = now + change
for name, step in [("do-nothing baseline", lambda s, a: s), ("simple model", simple_step), ("rich model", rich_step)]:
    err = np.mean((step(now, push) - real_next) ** 2)
    print(f"{name:>20}: one-step error on TEST episodes = {err:.6f}")

print("\nThe simple model is literally this 4×6 matrix of learned weights:")
print(np.round(W_simple, 3))

In [ ]:
quiz("compound")

---
## 4 · Imagination and compounding error 💭

To plan, a model must look several steps ahead **without seeing the real world**, feeding each guess back in as the next input.

### 🧩 Challenge 3 · Imagine the future

In [ ]:
def imagine(model_step, start_state, pushes):
    state = np.array(start_state, dtype=float)
    path = [state]
    for push in pushes:
        state = ___     # 🧩 the next state comes from the model's own guess
        path.append(state)
    return np.array(path)

imagine = check("imagine", imagine)

<details><summary>🤔 <b>Need a hint?</b></summary>

Call the model on the current (imagined) state and this push. That result becomes the new state.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>state = model_step(state, push)     # 🧩 the next state comes from the model's own guess</pre>

</details>

Now measure how prediction error grows with the number of imagined steps (the **horizon**). The y-axis is logarithmic: each gridline is 10× bigger.

In [ ]:
horizons = [1, 2, 5, 10, 20, 40]
def rollout_error(model_step, h):
    errors = []
    for ep in test_ids:
        for start in range(0, 45 - h, 5):
            imagined = imagine(model_step, states[ep, start], pushes[ep, start:start + h])
            errors.append(np.mean((imagined[-1] - states[ep, start + h]) ** 2))
    return np.mean(errors)

plt.figure(figsize=(6, 3.8))
for name, step in [("simple model", simple_step), ("rich model", rich_step)]:
    plt.plot(horizons, [rollout_error(step, h) for h in horizons], "o-", label=name)
plt.yscale("log"); plt.xlabel("steps imagined ahead"); plt.ylabel("squared error at the end")
plt.title("Small one-step mistakes compound"); plt.legend(); plt.show()

### 🎛️ Playground · Real vs imagined
Pick a test episode and how far to imagine. Black is what really happened. The dashed lines are what each model *imagined* from the same starting point with the same pushes.

In [ ]:
def compare(test_episode=0, horizon=20):
    ep = test_ids[test_episode]
    real = states[ep, :horizon + 1]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(real[:, 0], real[:, 1], "k-o", ms=3, label="real")
    for name, step, color in [("simple imagined", simple_step, "tab:orange"), ("rich imagined", rich_step, "tab:blue")]:
        path = imagine(step, states[ep, 0], pushes[ep, :horizon])
        ax.plot(path[:, 0], path[:, 1], "--", color=color, label=name)
    ax.plot(*real[0, :2], "gs", ms=9)
    ax.set(title=f"Test episode {test_episode}, {horizon} steps", xlabel="x", ylabel="y"); ax.legend(fontsize=8); plt.show()

playground(compare, test_episode=(0, 14, 1, 0), horizon=(1, 45, 1, 20))

---
## 5 · Plan by imagining 🎯

**Goal:** bring the puck to the centre `(0, 0)` and keep it there without wasting effort.

**Random shooting** works in three steps:
1. Invent many random 12-step push plans.
2. Imagine each one with the model and give it a **cost**: distance from the centre at every step, plus a small effort penalty.
3. Pick the cheapest plan.

The cost code below scores all plans **at once** (a "batch"), so it's fast.

In [ ]:
GOAL = np.array([0.0, 0.0])

def plan_costs(model_step, state, plans):
    # plans has shape (number_of_plans, horizon, 2)
    n, horizon, _ = plans.shape
    imagined = np.repeat(state[None], n, axis=0)        # every plan starts from the same real state
    cost = np.zeros(n)
    for t in range(horizon):
        imagined = model_step(imagined, plans[:, t])     # advance ALL plans one step together
        cost += np.sum((imagined[:, :2] - GOAL) ** 2, axis=1)    # far from goal = expensive
        cost += 0.02 * np.sum(plans[:, t] ** 2, axis=1)          # big pushes = a bit expensive
    return cost

### 🧩 Challenge 4 · Pick the best imagined plan

In [ ]:
def pick_best(costs):
    return ___     # 🧩 the index of the cheapest plan

pick_best = check("pick_best", pick_best)

<details><summary>🤔 <b>Need a hint?</b></summary>

You want the <i>position</i> of the smallest number in <code>costs</code>.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return int(np.argmin(costs))     # 🧩 the index of the cheapest plan</pre>

</details>

In [ ]:
quiz("mpc")

### 🧩 Challenge 5 · Act, then re-plan

In [ ]:
def first_push(best_plan):
    return ___          # 🧩 which part of the plan do we actually execute now?

first_push = check("first_push", first_push)

def random_shooting(model_step, state, gen, horizon=12, n_plans=192):
    plans = gen.uniform(-1, 1, (n_plans, horizon, 2))
    return plans[pick_best(plan_costs(model_step, state, plans))]

<details><summary>🤔 <b>Need a hint?</b></summary>

A plan is a list of 12 pushes. We only do the push for <i>right now</i>.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return best_plan[0]          # 🧩 which part of the plan do we actually execute now?</pre>

</details>

---
## 6 · Smarter search: the Cross-Entropy Method (CEM) 🔁

Random shooting wastes most samples on silly plans. **CEM** improves the guess over a few rounds:
1. Sample plans from a bell curve (a mean plan ± spread).
2. Keep the best 10%: the **elites**.
3. Move the bell curve to the elites' average and shrink its spread. Repeat.

### 🧩 Challenge 6 · Keep the elite plans

In [ ]:
def select_elites(plans, costs, n_elite):
    return ___    # 🧩 the n_elite cheapest plans

select_elites = check("elites", select_elites)

def cem(model_step, state, gen, horizon=12, n_plans=192, rounds=3):
    mean, spread = np.zeros((horizon, 2)), np.ones((horizon, 2))
    for _ in range(rounds):
        plans = np.clip(gen.normal(mean, spread, (n_plans, horizon, 2)), -1, 1)
        costs = plan_costs(model_step, state, plans)
        elites = select_elites(plans, costs, max(2, n_plans // 10))
        mean, spread = elites.mean(axis=0), np.maximum(elites.std(axis=0), 0.07)
    return plans[pick_best(costs)]

<details><summary>🤔 <b>Need a hint?</b></summary>

Sort indices by cost (lowest first), take the first <code>n_elite</code>, and use them to index <code>plans</code>.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return plans[np.argsort(costs)[:n_elite]]    # 🧩 the n_elite cheapest plans</pre>

</details>

### The control loop (MPC)
At every real step: plan with imagination → do the first push in the **real** world → observe the real result → plan again.

In [ ]:
START = np.array([-2.2, 1.5, 0.0, 0.0])    # far out on the bowl's slope, at rest

def run_controller(planner, model_step, start=START, steps=35, seed=9, **kw):
    gen = np.random.default_rng(seed)
    state, path, total_cost = start.copy(), [start.copy()], 0.0
    t0 = time.perf_counter()
    for _ in range(steps):
        if planner is None:
            push = gen.uniform(-1, 1, 2)                               # no planning: random pushing
        else:
            push = first_push(planner(model_step, state, gen, **kw))    # plan, then take the first push
        state = true_step(state, push)                                 # the REAL world responds
        path.append(state)
        total_cost += np.sum((state[:2] - GOAL) ** 2) + 0.02 * np.sum(push ** 2)
    return np.array(path), total_cost, (time.perf_counter() - t0) / steps

In [ ]:
quiz("perfect")

In [ ]:
contestants = [
    ("random pushing",               None,            None),
    ("random shooting · rich model", random_shooting, rich_step),
    ("CEM · simple model",           cem,             simple_step),
    ("CEM · rich model",             cem,             rich_step),
    ("CEM · TRUE physics (cheating)", cem,            true_step),
]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
print(f"{'controller':>32} | real total cost | ms per decision")
for name, planner, model in contestants:
    path, cost, sec = run_controller(planner, model)
    print(f"{name:>32} | {cost:15.2f} | {1000 * sec:8.1f}")
    ax.plot(path[:, 0], path[:, 1], "-o", ms=2, label=name)
ax.plot(0, 0, "r*", ms=16); ax.set(title="Who reaches the centre?", xlabel="x", ylabel="y"); ax.legend(fontsize=7); plt.show()

### 🎛️ Playground · Planning budget
More imagined plans and a longer horizon cost more compute. Do they always help? Watch **cost** and **milliseconds per decision**.

In [ ]:
def budget(n_plans=192, horizon=12, model="rich model"):
    model_step = {"rich model": rich_step, "simple model": simple_step, "true physics": true_step}[model]
    path, cost, sec = run_controller(cem, model_step, n_plans=n_plans, horizon=horizon)
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.plot(path[:, 0], path[:, 1], "-o", ms=2); ax.plot(0, 0, "r*", ms=14)
    ax.set(xlim=(-2.5, 2.5), ylim=(-2.5, 2.5), title=f"cost {cost:.2f} · {1000 * sec:.1f} ms/decision")
    plt.show()

playground(budget, n_plans=(8, 512, 8, 192), horizon=(1, 30, 1, 12), model=["rich model", "simple model", "true physics"])

---
## 7 · What if you can't re-plan? 🙈

Sometimes you **can't** check reality every step: a Mars rover with a 20-minute radio delay, or an agent that practises inside a learned simulator (Dreamer, NVIDIA Cosmos, DeepMind Genie). The model's imagination is then the only thing it has.

Below, each model plans **35 pushes once**, and we execute them blindly.

In [ ]:
quiz("blind")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
for ax, (name, model_step) in zip(axes, [("simple model", simple_step), ("rich model", rich_step)]):
    plan = cem(model_step, START, np.random.default_rng(9), horizon=35, n_plans=512, rounds=6)   # plan ONCE
    real = imagine(true_step, START, plan)            # what actually happens when we execute it
    dream = imagine(model_step, START, plan)          # what the model believed would happen
    ax.plot(dream[:, 0], dream[:, 1], "--", label="imagined")
    ax.plot(real[:, 0], real[:, 1], "-o", ms=2, label="real")
    ax.plot(0, 0, "r*", ms=14)
    miss = np.linalg.norm(real[-1, :2] - GOAL)
    ax.set(title=f"{name}: ends {miss:.2f} from the centre", xlabel="x", ylabel="y"); ax.legend(fontsize=8)
    print(f"{name:>13} · blind plan ends {miss:.3f} away from the goal")
plt.tight_layout(); plt.show()

---
## 8 · What you just built, and where it's used 🏭

| You did | The research name | Used in |
|---|---|---|
| Recorded random pushes | exploration / offline dataset | every robot-learning dataset (Open X-Embodiment, LeRobot) |
| Predicted state **change** | learned dynamics model | PETS, MBPO, Dreamer |
| Fed guesses back in | imagined rollout | DreamerV3 / Dreamer 4 learn behaviours inside imagination |
| Kept elite plans | CEM planning | Meta **V-JEPA 2-AC** plans robot arm motions with CEM in a learned feature space |
| Executed first push, re-planned | Model Predictive Control | self-driving planners, TD-MPC2, factory robots |

**The big lessons:**
* Re-planning from reality (MPC) forgives a lot. The simple model did fine in closed loop.
* When you must imagine far ahead **without feedback**, the model's accuracy is everything. The simple model's blind plan missed badly.

### 🧪 Try it yourself (HW3 core, optional)
1. In the planning-budget playground, try `horizon=1`, then `horizon=30`. What changes in the path and the cost, and why might looking further ahead matter for stopping a sliding puck?
2. Change the effort penalty `0.02` in `plan_costs` to `0.5`. What happens to the path?
3. **Uncertainty:** train 3 simple models on different random subsets of the training episodes. Where do their predictions disagree most? (Hint: far from the centre, where data is rare.)

### 🗣️ Explain it back
In two sentences, explain to a friend why a model that is 99% accurate one step ahead can still make a bad plan, and why re-planning helps.

In [ ]:
progress_report()